In [ ]:
# SETUP
from google.colab import drive
drive.mount('/content/drive')
# Install necessary packages (run once at the start of your notebook/script)
!pip install mne plotly colorama --upgrade
!pip install mne
# Import libraries
import os
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from colorama import Fore, Style, init
import plotly.graph_objects as go
import gc
import traceback

### RESEARCH QUESTION:  Can acoustic features from the distal context and critical region be used to predict whether a reduced function word will be perceived in continuous speech?

### Audio Data Loading and Inspection

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.io import wavfile
from glob import glob

# === 1. Root EEG/audio stimulus directory ===
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli'
file_paths = glob(os.path.join(root_dir, "*.wav"))

# === 2. Filter only files with _E.wav or _N.wav suffix ===
relevant_files = [f for f in file_paths if f.endswith("_E.wav") or f.endswith("_N.wav")]

# === 3. Extract metadata from each relevant file ===
audio_metadata = []

for path in relevant_files:
    sr, y = wavfile.read(path)

    # Normalize audio if not float
    if y.dtype != np.float32:
        y = y / np.max(np.abs(y))

    duration = len(y) / sr
    label = 'Expanded' if '_E' in path else 'Natural'

    audio_metadata.append({
        'filename': os.path.basename(path),
        'sampling_rate': sr,
        'duration_sec': duration,
        'label': label
    })

# === 4. Create and display metadata table ===
audio_df = pd.DataFrame(audio_metadata)
print("\n=== Audio Metadata ===")
print(audio_df)

# === 5. Plot waveforms for the first two files ===
for i, path in enumerate(relevant_files[:2]):
    sr, y = wavfile.read(path)
    y = y / np.max(np.abs(y))  # Normalize
    t = np.linspace(0, len(y) / sr, num=len(y))
    plt.figure(figsize=(12, 3))
    plt.plot(t, y, color='darkred')
    plt.title(f"Waveform: {os.path.basename(path)}")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.tight_layout()
    plt.show()


###Audio Preprocessing

| **Step** | **Task**                    | **Details**                                                                                                                                                                                   |
| -------- | --------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 2.1      | **Amplitude Normalization** | Converted raw audio samples to `float32` and scaled them to a range of $-1, 1$ to ensure consistent signal magnitude across all recordings.                                                   |
| 2.2      | **Silence Trimming**        | Removed leading and trailing low-energy frames using an energy threshold (`abs(y) > 0.02`) to eliminate padding and improve feature accuracy.                                                 |
| 2.3      | **Resampling**              | Resampled all audio signals to a consistent sample rate of **16,000 Hz** using `scipy.signal.resample` for uniform temporal resolution across files.                                          |
| 2.4      | **Label Assignment**        | Labeled files based on filename suffix: `_N.wav` = `1` (natural/fast speech, word likely perceived); `_E.wav` = `0` (expanded/slow speech, word less likely perceived), per ERP study design. |


In [ ]:
import os
import numpy as np
from scipy.io import wavfile
from scipy.signal import resample
from glob import glob
import pandas as pd

# === Step 2: Audio Preprocessing ===

# Set your folder path here
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli'
file_paths = glob(os.path.join(root_dir, "*.wav"))

# Filter only _E.wav or _N.wav files (exclude fillers or noise)
relevant_files = [f for f in file_paths if f.endswith("_E.wav") or f.endswith("_N.wav")]

# Set a target sample rate for consistency
target_sr = 16000

# Preprocessed data container
preprocessed_audio = []

for path in relevant_files:
    # === Step 2.1: Amplitude Normalization ===
    sr, y = wavfile.read(path)
    y = y.astype(np.float32)
    y = y / np.max(np.abs(y))  # Normalize to [-1, 1]

    # === Step 2.2: Resampling (if needed) ===
    if sr != target_sr:
        new_len = int(len(y) * target_sr / sr)
        y = resample(y, new_len)
        sr = target_sr

    # === Step 2.3: Silence Trimming ===
    energy = np.abs(y)
    threshold = 0.02
    mask = energy > threshold
    if np.any(mask):
        start = np.where(mask)[0][0]
        end = np.where(mask)[0][-1] + 1
        y = y[start:end]

    # === Step 2.4: Label Assignment ===
    label = 1 if "_N.wav" in path else 0  # 1 = Perceived (Natural), 0 = Not Perceived (Expanded)

    # Save preprocessed sample
    preprocessed_audio.append({
        'filename': os.path.basename(path),
        'audio': y,
        'sampling_rate': sr,
        'label': label
    })

# Optional summary DataFrame
summary_df = pd.DataFrame({
    'filename': [x['filename'] for x in preprocessed_audio],
    'samples': [len(x['audio']) for x in preprocessed_audio],
    'sampling_rate': [x['sampling_rate'] for x in preprocessed_audio],
    'duration_sec': [len(x['audio']) / x['sampling_rate'] for x in preprocessed_audio],
    'label': [x['label'] for x in preprocessed_audio]
})

# View or return the summary
print("✅ Preprocessing complete. Summary:")
print(summary_df.head())


### Feature Extraction

| **Step** | **Task**                        | **Details**                                                                      |
| -------- | ------------------------------- | -------------------------------------------------------------------------------- |
| 3.1      | Define feature extraction logic | Use `scipy` and `numpy` to extract features without `librosa`                    |
| 3.2      | Temporal Feature (ZCR)          | Count how often signal crosses zero — measures noisiness                         |
| 3.3      | Energy Feature (RMS)            | Measure average power of the signal                                              |
| 3.4      | Spectral Centroid               | Compute center of mass of the frequency spectrum — indicates brightness          |
| 3.5      | MFCC-like Features (1–13)       | Use Discrete Cosine Transform (DCT) of log spectrum as a cepstral representation |
| 3.6      | Assemble into DataFrame         | Final feature matrix with labels and filenames                                   |



In [ ]:
from scipy.signal import spectrogram
from scipy.fftpack import dct
import pandas as pd
import numpy as np

# === Step 3: Feature Extraction ===

# --- 3.1: Define feature extraction function ---
def extract_features_from_audio(y, sr):
    y = y.astype(np.float32)

    # --- 3.2: Temporal Feature: Zero Crossing Rate ---
    zcr = ((y[:-1] * y[1:]) < 0).sum() / len(y)

    # --- 3.3: Energy Feature: RMS ---
    rms = np.sqrt(np.mean(y ** 2))

    # --- 3.4: Spectral Feature: Spectral Centroid ---
    f, t, Sxx = spectrogram(y, fs=sr)
    Sxx_mean = np.mean(Sxx, axis=1)
    spectral_centroid = np.sum(f * Sxx_mean) / np.sum(Sxx_mean) if np.sum(Sxx_mean) > 0 else 0

    # --- 3.5: Cepstral Features: MFCC-like via DCT of log spectrum ---
    Sxx_log = np.log1p(Sxx_mean)  # log(1 + x) for numerical stability
    mfcc_like = dct(Sxx_log, type=2, norm='ortho')[:13]

    # Combine into dictionary
    features = {
        'zcr': zcr,
        'rms': rms,
        'spectral_centroid': spectral_centroid
    }
    for i in range(13):
        features[f'mfcc_{i+1}'] = mfcc_like[i]

    return features

# --- 3.6: Apply to all preprocessed audio ---
feature_rows = []
for sample in preprocessed_audio:
    y = sample['audio']
    sr = sample['sampling_rate']
    feats = extract_features_from_audio(y, sr)
    feats['filename'] = sample['filename']
    feats['label'] = sample['label']
    feature_rows.append(feats)

# Assemble final feature DataFrame
features_df = pd.DataFrame(feature_rows)

# Preview output
print("✅ Step 3 Complete — Extracted Features:")
print(features_df.head())


### Modeling & Evaluation (Random Forest)

| **Step** | **Task**                       | **Details**                                                            |
| -------- | ------------------------------ | ---------------------------------------------------------------------- |
| 4.1      | Prepare features and labels    | Separate input `X` (features) from `y` (labels); remove `filename`     |
| 4.2      | Train/Test split               | Use `train_test_split` with `stratify=y` to maintain label balance     |
| 4.3      | Feature scaling                | Standardize features using `StandardScaler` (zero mean, unit variance) |
| 4.4      | Train classifier               | Fit a `RandomForestClassifier` (fast, accurate, interpretable)         |
| 4.5      | Evaluate model                 | Print classification report and confusion matrix                       |
| 4.6      | Plot ROC curve and compute AUC | Visualize the model's discriminative ability                           |


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

# === Step 4: Modeling and Evaluation ===

# --- 4.1: Prepare Features and Labels ---
X = features_df.drop(columns=['filename', 'label'])
y = features_df['label']

# --- 4.2: Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# --- 4.3: Feature Scaling ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 4.4: Model Training ---
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# --- 4.5: Evaluation Metrics ---
y_pred = model.predict(X_test_scaled)
print("✅ Classification Report:")
print(classification_report(y_test, y_pred))
print("✅ Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# --- 4.6: ROC Curve and AUC ---
y_prob = model.predict_proba(X_test_scaled)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()


### Results Summary Table

| **Metric**               | **Value**          | **Interpretation**                                                |
| ------------------------ | ------------------ | ----------------------------------------------------------------- |
| **Test Accuracy**        | 71%                | Overall model correctly predicted 10 out of 14 test samples       |
| **Precision (Class 0)**  | 0.67               | Of predicted *not-perceived* (Expanded) samples, 67% were correct |
| **Recall (Class 0)**     | 0.67               | Model recovered 67% of actual *not-perceived* cases               |
| **Precision (Class 1)**  | 0.75               | Of predicted *perceived* (Natural) samples, 75% were correct      |
| **Recall (Class 1)**     | 0.75               | Model recovered 75% of actual *perceived* cases                   |
| **F1 Score (Macro Avg)** | 0.71               | Balanced F1 across both classes                                   |
| **Confusion Matrix**     | `[[4, 2], [2, 6]]` | 4 TN, 6 TP, 2 FP, 2 FN — modest misclassification on both sides   |
| **AUC (ROC Curve)**      | **0.82**           | Model has strong ability to distinguish between classes (0 vs 1)  |


### Interpretation


*   The model shows balanced precision and recall across both labels.
*   AUC = 0.82 is excellent for a small dataset — the classifier learned meaningful patterns from MFCCs and acoustic features.





### Modeling & Evaluation (XGBoost)

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# === Step 4b: XGBoost Classifier ===

# 4b.1: Prepare features and labels
X = features_df.drop(columns=['filename', 'label'])
y = features_df['label']

# 4b.2: Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 4b.3: Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4b.4: Train XGBoost Classifier
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_scaled, y_train)

# 4b.5: Evaluation
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_prob_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Classification report
print("✅ XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))

# Confusion matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print("✅ Confusion Matrix:")
print(cm_xgb)

# 4b.6: ROC Curve and AUC
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
roc_auc_xgb = auc(fpr_xgb, tpr_xgb)

# Plot ROC
plt.figure(figsize=(6, 5))
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost ROC (AUC = {roc_auc_xgb:.2f})")
plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("XGBoost ROC Curve")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()


### Result Summary Table (XGBoost)

| **Metric**               | **Value**          | **Interpretation**                                            |
| ------------------------ | ------------------ | ------------------------------------------------------------- |
| **Accuracy**             | 64%                | Correctly predicted 9 of 14 test samples                      |
| **Precision (Class 0)**  | 0.57               | Of predicted "Not Perceived" (Expanded), 57% were correct     |
| **Recall (Class 0)**     | 0.67               | Detected 67% of true class 0 cases                            |
| **Precision (Class 1)**  | 0.71               | Of predicted "Perceived" (Natural), 71% were correct          |
| **Recall (Class 1)**     | 0.62               | Detected 62% of true class 1 cases                            |
| **F1 Score (Macro Avg)** | 0.65               | Balanced F1 score across classes                              |
| **Confusion Matrix**     | `[[4, 2], [3, 5]]` | Slightly more false negatives than Random Forest              |
| **AUC (ROC Curve)**      | **0.77**           | Strong separability; slightly lower than Random Forest (0.82) |


 ### Random Forest vs. XGBoost – Performance Comparison

| **Metric**               | **Random Forest**  | **XGBoost**        | **Observation**                          |
| ------------------------ | ------------------ | ------------------ | ---------------------------------------- |
| **Accuracy**             | **71%**            | 64%                | RF performs better overall               |
| **Precision (Class 0)**  | 0.67               | 0.57               | RF has fewer false positives for class 0 |
| **Recall (Class 0)**     | 0.67               | **0.67**           | Equal ability to identify class 0        |
| **Precision (Class 1)**  | 0.75               | 0.71               | Both are solid; RF slightly better       |
| **Recall (Class 1)**     | **0.75**           | 0.62               | RF detects more "Perceived" cases        |
| **F1 Score (Macro Avg)** | **0.71**           | 0.65               | RF gives more balanced performance       |
| **AUC (ROC Curve)**      | **0.82**           | 0.77               | RF is better at class separation         |
| **Confusion Matrix**     | `[[4, 2], [2, 6]]` | `[[4, 2], [3, 5]]` | XGBoost missed 1 more "Perceived" case   |


### Model Selection:

| **Criteria**                | **Recommended Model** | **Why**                                                    |
| --------------------------- | --------------------- | ---------------------------------------------------------- |
| Best baseline accuracy      | ✅ **Random Forest**   | Higher accuracy and AUC without tuning                     |
| Interpretability            | ✅ **Random Forest**   | Easier to explain with built-in feature importance         |
| Generalization (small data) | ✅ **Random Forest**   | More stable without hyperparameter tuning                  |
| Requires fine-tuning        | ⚠️ XGBoost            | Needs grid search or cross-validation for best performance |


### Feature Importance & Interpretation (Random Forest)



| **Step** | **Task**                   | **Details**                                                                                              |
| -------- | -------------------------- | -------------------------------------------------------------------------------------------------------- |
| **5.1**  | Extract feature importance | Use `.feature_importances_` from `RandomForestClassifier` to compute relative importance of each feature |
| **5.2**  | Rank and visualize         | Rank features and visualize using a bar plot                                                             |
| **5.3**  | Interpret key drivers      | Identify which acoustic features (e.g., MFCCs, ZCR, RMS) are driving perception classification           |


### Feature Importance Random Forest

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# === Step 5: Feature Importance ===

# --- 5.1: Extract Feature Importances ---
importances = model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

# --- 5.2: Plot Feature Importances ---
plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'], color='darkred')
plt.gca().invert_yaxis()  # Highest importance at top
plt.title("Feature Importances (Random Forest)")
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.show()

# --- 5.3: Output ranked feature list ---
print("✅ Top Audio Features Driving Classification:")
print(importance_df.head(10))


### Feature Importance Results & Interpretation

| **Rank** | **Feature**         | **Importance Score** | **Interpretation**                                                                                                                                 |
| -------- | ------------------- | -------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1        | `mfcc_13`           | 0.1297               | Higher-order MFCC capturing subtle timbre/energy shifts at sentence end; possibly encoding prosodic cues of speech rate or final syllable strength |
| 2        | `mfcc_12`           | 0.1275               | Captures late spectral details — often influenced by consonant/vowel structure transitions in reduced function words                               |
| 3        | `mfcc_11`           | 0.0972               | Mid-to-high frequency emphasis; might relate to stressed syllable detection or consonant energy                                                    |
| 4        | `mfcc_10`           | 0.0748               | Related to articulation clarity in midband formants                                                                                                |
| 5        | `zcr`               | 0.0707               | Noisiness — reduced words often have fewer zero crossings due to weaker consonants                                                                 |
| 6        | `mfcc_9`            | 0.0633               | Phonetic boundary softness or vowel nasalization may affect this                                                                                   |
| 7        | `spectral_centroid` | 0.0622               | Lower centroid → slower speech; helps distinguish Expanded from Natural                                                                            |
| 8        | `rms`               | 0.0584               | Loudness; reduced words often have lower amplitude                                                                                                 |
| 9        | `mfcc_1`            | 0.0560               | Represents overall spectral envelope energy; foundational                                                                                          |
| 10       | `mfcc_4`            | 0.0501               | Captures lower-to-mid frequency formant transitions                                                                                                |




*   Higher MFCCs (10–13) are most important, likely capturing fine-grained spectral dynamics from prosody, vowel reduction, or syllable timing.

*  ZCR and Spectral Centroid confirm the model is sensitive to speech clarity and pacing.

*   RMS shows that energy plays a role — function words tend to be less energetic when reduced.



### Feature Importance & Interpretation (XGBoost)



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# === Step 5: Feature Importance (XGBoost) ===

# --- 5.1: Extract Feature Importances ---
importances = xgb_model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

# --- 5.2: Plot Feature Importances ---
plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'], color='darkgreen')
plt.gca().invert_yaxis()  # Highest importance at top
plt.title("Feature Importances (XGBoost)")
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.show()

# --- 5.3: Output ranked feature list ---
print("✅ Top Audio Features Driving Classification (XGBoost):")
print(importance_df.head(10))



| **Feature**         | **Importance** | **What It Means**                                                                                                                                                       |
| ------------------- | -------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `mfcc_13`           | **0.157**      | Final cepstral coefficient; captures **subtle decay and energy dispersion** — strong cue for distinguishing **reduced words** (expanded speech ends softer and flatter) |
| `mfcc_12`           | 0.138          | Near-final spectral dynamics — captures **fine prosodic timing**, especially in **unstressed syllables**                                                                |
| `spectral_centroid` | 0.126          | Represents **"brightness" or average frequency** — higher in faster, natural speech; **lower in expanded (slower)**                                                     |
| `rms`               | 0.106          | Loudness; **reduced words tend to be quieter**, expanded ones more prolonged and louder                                                                                 |
| `mfcc_10`           | 0.101          | Related to **mid-band frequency shape**, possibly encoding **vowel quality or clarity**                                                                                 |
| `mfcc_8`            | 0.079          | Spectral pattern variations from mid-range — linked to **coarticulation or reduced articulation**                                                                       |
| `mfcc_11`           | 0.067          | Captures **phonetic transitions** — may detect **unstressed function words**                                                                                            |
| `zcr`               | 0.056          | Measures **noisiness or sharpness** — reduced speech may have **fewer zero crossings**                                                                                  |
| `mfcc_5`, `mfcc_2`  | \~0.04         | Base spectral envelope components; contribute **moderately** to identifying formant structure                                                                           |


### Comparison between the Random Forest and XGBoost

| **Rank** | **Feature**                    | **Importance (XGBoost)** | **Importance (Random Forest)** | **Interpretation**                                                                  |
| -------- | ------------------------------ | ------------------------ | ------------------------------ | ----------------------------------------------------------------------------------- |
| 1        | `mfcc_13`                      | **0.157**                | **0.130**                      | High-order cepstral feature; encodes fine-grained spectral decay at word boundaries |
| 2        | `mfcc_12`                      | 0.138                    | 0.127                          | Subtle changes in energy/timbre — often reflect reduced syllables                   |
| 3        | `spectral_centroid`            | 0.126                    | 0.062                          | Brightness/frequency balance; faster speech shifts centroid higher                  |
| 4        | `rms`                          | 0.106                    | 0.058                          | Energy amplitude; reduced function words = lower RMS                                |
| 5        | `mfcc_10`                      | 0.101                    | 0.075                          | Mid-to-high frequency envelope; links to clarity/stress                             |
| 6        | `mfcc_8`                       | 0.079                    | —                              | Present only in XGBoost; likely contributes to formant contour modeling             |
| 7        | `mfcc_11`                      | 0.067                    | 0.097                          | Consistent relevance; tied to consonant transitions                                 |
| 8        | `zcr`                          | 0.056                    | 0.071                          | Measures noisiness; reduced words tend to have lower ZCR                            |
| 9        | `mfcc_5` (XGB) / `mfcc_9` (RF) | 0.046 / —                | — / 0.063                      | Both affect spectral detail; possible articulation cues                             |
| 10       | `mfcc_2` (XGB) / `mfcc_1` (RF) | 0.038 / —                | — / 0.056                      | Foundational MFCCs capturing general spectral shape                                 |


After training both Random Forest and XGBoost classifiers on MFCC and prosodic features, I performed feature importance analysis. This revealed that MFCC13 and spectral centroid were the strongest predictors of reduced function word perception — confirming that spectral decay and pacing cues drive listener interpretation